
# Buy Fee Model V13 — Risk-Based Decision Model

## Purpose

V13 turns the strongest existing evidence from V8 / V10 / V11 into a practical pricing decision framework.

It does **not** attempt to predict an exact future RPU value with false precision.

Instead, for each:

**Location × Lane × Price Tier × Fee Scenario**

and for:

- +$10
- +$15
- +$20

V13 outputs:

- historical matched-control support
- shrunk expected RPU effect
- empirical RPU range
- Conversion downside risk
- Volume downside risk
- event count
- independent corporate change-date count
- segment confidence
- evidence strength
- decision-support status

## Data hierarchy

V13 uses:

### V8.2
Matched-control Difference-in-Differences event outcomes.

### V10.1
Portfolio-level diagnostics showing which drivers matter most.

### V11
Scenario-specific shrunk segment responses and validation results.

## Philosophy

The final output should answer:

> "For this segment, what does the historical evidence suggest about a +$10 / +$15 / +$20 increase, and how much confidence should we place in that evidence?"

rather than:

> "The exact future RPU will be $17.83."


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"
V101_DIR = DATA_FOLDER / "buy_fee_model_v10_1_outputs"
V11_DIR = DATA_FOLDER / "buy_fee_model_v11_outputs"

V8_EVENT_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

V10_DRIVER_FILE = V101_DIR / "14_DRIVER_DIAGNOSTIC_RANKING.csv"
V10_SCENARIO_FILE = V101_DIR / "15_SCENARIO_SUCCESS_DIAGNOSTICS.csv"

V11_SEGMENT_FILE = V11_DIR / "07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv"
V11_VALIDATION_FILE = V11_DIR / "04_SCENARIO_MODEL_VALIDATION.csv"
V11_READINESS_FILE = V11_DIR / "08_V11_READINESS_SUMMARY.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v13_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20]
TOLERANCE = 2.5

# ------------------------------------------------------------
# Risk thresholds
# ------------------------------------------------------------

# Conversion risk uses shrunk pp effect
CONVERSION_LOW_MIN = -1.0
CONVERSION_MODERATE_MIN = -3.0

# Volume risk uses shrunk percent effect
VOLUME_LOW_MIN = -0.03
VOLUME_MODERATE_MIN = -0.10

# ------------------------------------------------------------
# Evidence-strength thresholds
# ------------------------------------------------------------

HIGH_EVENTS = 25
HIGH_DATES = 3

MEDIUM_EVENTS = 10
MEDIUM_DATES = 2

# Portfolio support thresholds
PORTFOLIO_HIGH_DATES = 10
PORTFOLIO_MEDIUM_DATES = 5

print("=" * 80)
print("V13 INPUT CHECK")
print("=" * 80)

required_files = [
    V8_EVENT_FILE,
    V11_SEGMENT_FILE,
]

for p in required_files:
    print(f"{p.name:<45} {p.exists()}")

missing = [p for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required V13 input(s):\n"
        + "\n".join(str(p) for p in missing)
    )

print("\nOptional files:")
for p in [
    V10_DRIVER_FILE,
    V10_SCENARIO_FILE,
    V11_VALIDATION_FILE,
    V11_READINESS_FILE,
]:
    print(f"{p.name:<45} {p.exists()}")

print("\nOutput folder:")
print(OUTPUT_DIR)


V13 INPUT CHECK
04_DID_EVENT_LEVEL_RESULTS.csv                True
07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv       True

Optional files:
14_DRIVER_DIAGNOSTIC_RANKING.csv              True
15_SCENARIO_SUCCESS_DIAGNOSTICS.csv           True
04_SCENARIO_MODEL_VALIDATION.csv              True
08_V11_READINESS_SUMMARY.csv                  True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v13_outputs


## 1. Load V8 matched-control event evidence

In [2]:

v8 = pd.read_csv(
    V8_EVENT_FILE,
    low_memory=True
)

for c in [
    "BUYFEE_CHANGE_AMOUNT",
    "DiD_RPU_Change_Dollar",
    "DiD_Conversion_Change_pp",
    "DiD_Volume_Change_Pct",
]:
    if c in v8.columns:
        v8[c] = pd.to_numeric(
            v8[c],
            errors="coerce"
        )

required = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "DiD_RPU_Change_Dollar",
    "DiD_Conversion_Change_pp",
    "DiD_Volume_Change_Pct",
]

missing = [c for c in required if c not in v8.columns]

if missing:
    raise ValueError(
        "V8 event file missing required columns:\n"
        + "\n".join(missing)
    )

print("V8 events:", f"{len(v8):,}")
print("Locations:", v8["Treated_Location"].nunique())
print("Corporate change dates:", v8["Corporate_Change_ID"].nunique())


V8 events: 1,824
Locations: 35
Corporate change dates: 37


## 2. Build portfolio scenario evidence from V8

In [3]:

portfolio_rows = []

for fee in SCENARIOS:

    d = v8[
        v8["BUYFEE_CHANGE_AMOUNT"].between(
            fee - TOLERANCE,
            fee + TOLERANCE
        )
    ].copy()

    rpu = pd.to_numeric(
        d["DiD_RPU_Change_Dollar"],
        errors="coerce"
    ).dropna()

    conv = pd.to_numeric(
        d["DiD_Conversion_Change_pp"],
        errors="coerce"
    ).dropna()

    vol = pd.to_numeric(
        d["DiD_Volume_Change_Pct"],
        errors="coerce"
    ).dropna()

    portfolio_rows.append({
        "Fee_Increase": fee,

        "Portfolio_Events": len(d),
        "Portfolio_Locations": d["Treated_Location"].nunique(),
        "Portfolio_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Portfolio_Median_RPU": rpu.median(),
        "Portfolio_RPU_P25": rpu.quantile(0.25),
        "Portfolio_RPU_P75": rpu.quantile(0.75),

        "Portfolio_Median_Conversion_pp": conv.median(),
        "Portfolio_Conversion_P25": conv.quantile(0.25),
        "Portfolio_Conversion_P75": conv.quantile(0.75),

        "Portfolio_Median_Volume_Pct": vol.median(),
        "Portfolio_Volume_P25": vol.quantile(0.25),
        "Portfolio_Volume_P75": vol.quantile(0.75),

        "P_RPU_Positive": (
            (rpu > 0).mean()
            if len(rpu)
            else np.nan
        ),

        "P_Conversion_Down_GT_2pp": (
            (conv < -2).mean()
            if len(conv)
            else np.nan
        ),

        "P_Volume_Down_GT_10pct": (
            (vol < -0.10).mean()
            if len(vol)
            else np.nan
        ),
    })

portfolio = pd.DataFrame(
    portfolio_rows
)

portfolio.to_csv(
    OUTPUT_DIR / "01_PORTFOLIO_SCENARIO_EVIDENCE.csv",
    index=False
)

display(portfolio)


,Fee_Increase,Portfolio_Events,Portfolio_Locations,Portfolio_Change_Dates,Portfolio_Median_RPU,Portfolio_RPU_P25,Portfolio_RPU_P75,Portfolio_Median_Conversion_pp,Portfolio_Conversion_P25,Portfolio_Conversion_P75,Portfolio_Median_Volume_Pct,Portfolio_Volume_P25,Portfolio_Volume_P75,P_RPU_Positive,P_Conversion_Down_GT_2pp,P_Volume_Down_GT_10pct
0,10,594,16,24,7.630647,-12.837704,31.211050,0.627338,-7.079622,9.118519,-0.003620,-0.209742,0.217375,0.594276,0.414141,0.388889
1,15,84,14,10,10.481982,-13.684255,27.833303,1.790793,-9.233940,11.779638,0.044703,-0.204540,0.257155,0.630952,0.392857,0.345238
2,20,133,12,10,24.466728,1.208639,54.571618,1.658328,-7.014595,8.818850,-0.025353,-0.258778,0.215278,0.759398,0.413534,0.406015


## 3. Load V11 shrunk segment responses

In [4]:

segments = pd.read_csv(
    V11_SEGMENT_FILE,
    low_memory=True
)

required_seg = [
    "Fee_Increase",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "Events",
    "Change_Dates",
    "Shrunk_RPU_Effect",
    "Shrunk_Conversion_Effect",
    "Shrunk_Volume_Effect",
    "Segment_Confidence",
]

missing = [
    c for c in required_seg
    if c not in segments.columns
]

if missing:
    raise ValueError(
        "V11 segment file missing required columns:\n"
        + "\n".join(missing)
    )

for c in [
    "Fee_Increase",
    "Events",
    "Change_Dates",
    "Shrunk_RPU_Effect",
    "Shrunk_Conversion_Effect",
    "Shrunk_Volume_Effect",
]:
    segments[c] = pd.to_numeric(
        segments[c],
        errors="coerce"
    )

segments = segments[
    segments["Fee_Increase"].isin(
        SCENARIOS
    )
].copy()

print("Scenario segment rows:", f"{len(segments):,}")
print("Locations:", segments["Treated_Location"].nunique())


Scenario segment rows: 603
Locations: 24


## 4. Add model validation context from V11

In [5]:

if V11_VALIDATION_FILE.exists():

    validation = pd.read_csv(
        V11_VALIDATION_FILE,
        low_memory=True
    )

    validation_small = validation[
        validation["Fee_Increase"].isin(
            SCENARIOS
        )
    ].copy()

else:

    validation_small = pd.DataFrame()

if V11_READINESS_FILE.exists():

    readiness = pd.read_csv(
        V11_READINESS_FILE,
        low_memory=True
    )

else:

    readiness = pd.DataFrame()

print("Validation rows:", len(validation_small))
print("Readiness rows :", len(readiness))


Validation rows: 9
Readiness rows : 9


## 5. Risk classification functions

In [6]:

def conversion_risk(effect_pp):

    if pd.isna(effect_pp):
        return "Unknown"

    if effect_pp >= CONVERSION_LOW_MIN:
        return "Low"

    if effect_pp >= CONVERSION_MODERATE_MIN:
        return "Moderate"

    return "High"


def volume_risk(effect_pct):

    if pd.isna(effect_pct):
        return "Unknown"

    if effect_pct >= VOLUME_LOW_MIN:
        return "Low"

    if effect_pct >= VOLUME_MODERATE_MIN:
        return "Moderate"

    return "High"


def segment_evidence_strength(
    events,
    change_dates,
    segment_confidence
):

    if (
        events >= HIGH_EVENTS
        and change_dates >= HIGH_DATES
        and str(segment_confidence).upper() == "HIGH"
    ):
        return "High"

    if (
        events >= MEDIUM_EVENTS
        and change_dates >= MEDIUM_DATES
    ):
        return "Medium"

    return "Limited"


def portfolio_evidence_strength(change_dates):

    if change_dates >= PORTFOLIO_HIGH_DATES:
        return "High"

    if change_dates >= PORTFOLIO_MEDIUM_DATES:
        return "Medium"

    return "Limited"


def decision_status(
    evidence,
    conversion_risk_value,
    volume_risk_value
):

    if evidence == "Limited":
        return "RESEARCH / TEST ONLY"

    if (
        conversion_risk_value == "High"
        or volume_risk_value == "High"
    ):
        return "HIGH RISK - REVIEW"

    if (
        evidence == "High"
        and conversion_risk_value == "Low"
        and volume_risk_value == "Low"
    ):
        return "SUPPORTED FOR PILOT REVIEW"

    return "REVIEW WITH GUARDRAILS"


## 6. Merge segment and portfolio evidence

In [7]:

decision = segments.merge(
    portfolio,
    on="Fee_Increase",
    how="left"
)

decision["Conversion_Risk"] = (
    decision[
        "Shrunk_Conversion_Effect"
    ].apply(
        conversion_risk
    )
)

decision["Volume_Risk"] = (
    decision[
        "Shrunk_Volume_Effect"
    ].apply(
        volume_risk
    )
)

decision["Segment_Evidence_Strength"] = (
    decision.apply(
        lambda r:
        segment_evidence_strength(
            r["Events"],
            r["Change_Dates"],
            r["Segment_Confidence"]
        ),
        axis=1
    )
)

decision["Portfolio_Evidence_Strength"] = (
    decision[
        "Portfolio_Change_Dates"
    ].apply(
        portfolio_evidence_strength
    )
)

# Conservative combined evidence:
# use the weaker of segment and portfolio evidence
evidence_order = {
    "Limited": 0,
    "Medium": 1,
    "High": 2,
}

reverse_evidence = {
    0: "Limited",
    1: "Medium",
    2: "High",
}

decision["Evidence_Strength"] = (
    decision.apply(
        lambda r:
        reverse_evidence[
            min(
                evidence_order.get(
                    r["Segment_Evidence_Strength"],
                    0
                ),
                evidence_order.get(
                    r["Portfolio_Evidence_Strength"],
                    0
                )
            )
        ],
        axis=1
    )
)

decision["Decision_Status"] = (
    decision.apply(
        lambda r:
        decision_status(
            r["Evidence_Strength"],
            r["Conversion_Risk"],
            r["Volume_Risk"]
        ),
        axis=1
    )
)


## 7. Create practical RPU ranges

In [8]:

# ============================================================
# The segment's shrunk effect is the center.
# Portfolio interquartile spread is used as an uncertainty band.
#
# This is intentionally a range, not a falsely precise forecast.
# ============================================================

decision[
    "Expected_RPU_Center"
] = (
    decision[
        "Shrunk_RPU_Effect"
    ]
)

decision[
    "Expected_RPU_Low"
] = (
    decision[
        "Shrunk_RPU_Effect"
    ]
    +
    (
        decision[
            "Portfolio_RPU_P25"
        ]
        -
        decision[
            "Portfolio_Median_RPU"
        ]
    )
)

decision[
    "Expected_RPU_High"
] = (
    decision[
        "Shrunk_RPU_Effect"
    ]
    +
    (
        decision[
            "Portfolio_RPU_P75"
        ]
        -
        decision[
            "Portfolio_Median_RPU"
        ]
    )
)

# Ensure sensible ordering
rpu_low = decision[
    [
        "Expected_RPU_Low",
        "Expected_RPU_High"
    ]
].min(axis=1)

rpu_high = decision[
    [
        "Expected_RPU_Low",
        "Expected_RPU_High"
    ]
].max(axis=1)

decision[
    "Expected_RPU_Low"
] = rpu_low

decision[
    "Expected_RPU_High"
] = rpu_high


decision[
    "RPU_Range_Interpretation"
] = np.select(
    [
        decision[
            "Expected_RPU_Low"
        ] > 0,

        decision[
            "Expected_RPU_High"
        ] < 0,
    ],
    [
        "Range remains positive",
        "Range remains negative",
    ],
    default="Range crosses zero"
)


## 8. Add validation warning flags

In [9]:

# Pull RPU validation by scenario when available
if not validation_small.empty:

    rpu_val = validation_small[
        validation_small["Target"].astype(str).str.upper().eq(
            "RPU"
        )
    ].copy()

    rpu_keep = [
        c for c in [
            "Fee_Increase",
            "Median_R2",
            "Median_MAE",
            "Median_Direction_Accuracy",
            "Positive_R2_Fold_Share",
            "Folds",
        ]
        if c in rpu_val.columns
    ]

    rpu_val = rpu_val[
        rpu_keep
    ].drop_duplicates(
        subset=["Fee_Increase"]
    )

    decision = decision.merge(
        rpu_val,
        on="Fee_Increase",
        how="left"
    )

else:

    decision["Median_R2"] = np.nan
    decision["Median_MAE"] = np.nan
    decision["Median_Direction_Accuracy"] = np.nan
    decision["Positive_R2_Fold_Share"] = np.nan
    decision["Folds"] = np.nan


decision[
    "Prediction_Quality_Flag"
] = np.select(
    [
        (
            decision[
                "Median_Direction_Accuracy"
            ] >= 0.65
        )
        &
        (
            decision[
                "Median_R2"
            ] > 0
        ),

        (
            decision[
                "Median_Direction_Accuracy"
            ] >= 0.60
        ),
    ],
    [
        "Promising",
        "Directional Only"
    ],
    default="Weak"
)


## 9. Final segment decision table

In [10]:

final_cols = [
    "Fee_Increase",

    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",

    "Events",
    "Change_Dates",

    "Expected_RPU_Center",
    "Expected_RPU_Low",
    "Expected_RPU_High",
    "RPU_Range_Interpretation",

    "Shrunk_Conversion_Effect",
    "Conversion_Risk",

    "Shrunk_Volume_Effect",
    "Volume_Risk",

    "P_RPU_Positive",
    "P_Conversion_Down_GT_2pp",
    "P_Volume_Down_GT_10pct",

    "Segment_Confidence",
    "Segment_Evidence_Strength",
    "Portfolio_Evidence_Strength",
    "Evidence_Strength",

    "Median_R2",
    "Median_MAE",
    "Median_Direction_Accuracy",
    "Prediction_Quality_Flag",

    "Decision_Status",
]

final_cols = [
    c for c in final_cols
    if c in decision.columns
]

final = decision[
    final_cols
].copy()

final = final.sort_values(
    [
        "Fee_Increase",
        "Evidence_Strength",
        "Treated_Location",
        "Treated_Lane",
        "Treated_Price_Tier"
    ],
    ascending=[
        True,
        True,
        True,
        True,
        True
    ]
)

final.to_csv(
    OUTPUT_DIR
    / "02_V13_SEGMENT_DECISION_TABLE.csv",
    index=False
)

display(
    final.head(30)
)


,Fee_Increase,Treated_Location,Treated_Lane,Treated_Price_Tier,Events,Change_Dates,Expected_RPU_Center,Expected_RPU_Low,Expected_RPU_High,RPU_Range_Interpretation,...,P_Volume_Down_GT_10pct,Segment_Confidence,Segment_Evidence_Strength,Portfolio_Evidence_Strength,Evidence_Strength,Median_R2,Median_MAE,Median_Direction_Accuracy,Prediction_Quality_Flag,Decision_Status
0,10,ATLANTA,IL,0-1000,1,1,6.965769,-13.502582,30.546172,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
1,10,ATLANTA,IO,0-1000,1,1,9.756983,-10.711368,33.337386,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
2,10,BATONROUGE,1,1000-2000,1,1,6.127884,-14.340468,29.708286,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
3,10,BATONROUGE,3,1000-2000,1,1,8.436959,-12.031392,32.017362,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
4,10,BATONROUGE,5,1000-2000,1,1,6.230580,-14.237772,29.810982,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
5,10,BOSTON,A,0-1000,1,1,7.914420,-12.553932,31.494822,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
6,10,BOSTON,A,1000-2000,1,1,7.319381,-13.148970,30.899783,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
7,10,BOSTON,A,10000-11000,1,1,11.391347,-9.077004,34.971750,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
8,10,BOSTON,A,11000-12000,1,1,8.308741,-12.159610,31.889143,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY
9,10,BOSTON,A,12000-13000,1,1,7.314147,-13.154205,30.894549,Range crosses zero,...,0.388889,Low,Limited,High,Limited,-0.411441,31.497519,0.642857,Directional Only,RESEARCH / TEST ONLY


## 10. Portfolio scenario summary

In [11]:

portfolio_decision = (
    final.groupby(
        "Fee_Increase",
        as_index=False
    )
    .agg(
        Segments=(
            "Treated_Location",
            "size"
        ),

        Locations=(
            "Treated_Location",
            "nunique"
        ),

        Median_Expected_RPU=(
            "Expected_RPU_Center",
            "median"
        ),

        Median_RPU_Low=(
            "Expected_RPU_Low",
            "median"
        ),

        Median_RPU_High=(
            "Expected_RPU_High",
            "median"
        ),

        Low_Conversion_Risk_Share=(
            "Conversion_Risk",
            lambda s:
            (
                s == "Low"
            ).mean()
        ),

        High_Conversion_Risk_Share=(
            "Conversion_Risk",
            lambda s:
            (
                s == "High"
            ).mean()
        ),

        Low_Volume_Risk_Share=(
            "Volume_Risk",
            lambda s:
            (
                s == "Low"
            ).mean()
        ),

        High_Volume_Risk_Share=(
            "Volume_Risk",
            lambda s:
            (
                s == "High"
            ).mean()
        ),

        High_Evidence_Segments=(
            "Evidence_Strength",
            lambda s:
            (
                s == "High"
            ).sum()
        ),

        Medium_Evidence_Segments=(
            "Evidence_Strength",
            lambda s:
            (
                s == "Medium"
            ).sum()
        ),

        Limited_Evidence_Segments=(
            "Evidence_Strength",
            lambda s:
            (
                s == "Limited"
            ).sum()
        ),

        Supported_Pilot_Segments=(
            "Decision_Status",
            lambda s:
            (
                s == "SUPPORTED FOR PILOT REVIEW"
            ).sum()
        ),
    )
)

portfolio_decision = portfolio_decision.merge(
    portfolio,
    on="Fee_Increase",
    how="left"
)

portfolio_decision.to_csv(
    OUTPUT_DIR
    / "03_V13_PORTFOLIO_SCENARIO_SUMMARY.csv",
    index=False
)

display(portfolio_decision)


,Fee_Increase,Segments,Locations,Median_Expected_RPU,Median_RPU_Low,Median_RPU_High,Low_Conversion_Risk_Share,High_Conversion_Risk_Share,Low_Volume_Risk_Share,High_Volume_Risk_Share,...,Portfolio_RPU_P75,Portfolio_Median_Conversion_pp,Portfolio_Conversion_P25,Portfolio_Conversion_P75,Portfolio_Median_Volume_Pct,Portfolio_Volume_P25,Portfolio_Volume_P75,P_RPU_Positive,P_Conversion_Down_GT_2pp,P_Volume_Down_GT_10pct
0,10,390,16,8.789637,-11.678715,32.370039,0.994872,0.0,1.000000,0.0,...,31.211050,0.627338,-7.079622,9.118519,-0.003620,-0.209742,0.217375,0.594276,0.414141,0.388889
1,15,84,14,5.900824,-18.265413,23.252145,1.000000,0.0,1.000000,0.0,...,27.833303,1.790793,-9.233940,11.779638,0.044703,-0.204540,0.257155,0.630952,0.392857,0.345238
2,20,129,12,30.761922,7.503832,60.866812,1.000000,0.0,0.976744,0.0,...,54.571618,1.658328,-7.014595,8.818850,-0.025353,-0.258778,0.215278,0.759398,0.413534,0.406015


## 11. Evidence-strength distribution

In [12]:

evidence_summary = (
    final.groupby(
        [
            "Fee_Increase",
            "Evidence_Strength",
            "Decision_Status"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="Segments"
    )
)

evidence_summary.to_csv(
    OUTPUT_DIR
    / "04_V13_EVIDENCE_STRENGTH_SUMMARY.csv",
    index=False
)

display(evidence_summary)


,Fee_Increase,Evidence_Strength,Decision_Status,Segments
0,10,Limited,RESEARCH / TEST ONLY,390
1,15,Limited,RESEARCH / TEST ONLY,84
2,20,Limited,RESEARCH / TEST ONLY,129


## 12. Best-supported pilot candidates

In [13]:

# This is NOT an automatic pricing recommendation.
# It is a shortlist of historically better-supported segments
# for human review / controlled testing.

pilot = final[
    final[
        "Decision_Status"
    ].isin(
        [
            "SUPPORTED FOR PILOT REVIEW",
            "REVIEW WITH GUARDRAILS"
        ]
    )
].copy()

risk_order = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Unknown": 3,
}

evidence_rank = {
    "High": 0,
    "Medium": 1,
    "Limited": 2,
}

pilot["_Evidence_Order"] = (
    pilot[
        "Evidence_Strength"
    ].map(
        evidence_rank
    ).fillna(9)
)

pilot["_Conv_Risk_Order"] = (
    pilot[
        "Conversion_Risk"
    ].map(
        risk_order
    ).fillna(9)
)

pilot["_Vol_Risk_Order"] = (
    pilot[
        "Volume_Risk"
    ].map(
        risk_order
    ).fillna(9)
)

pilot = pilot.sort_values(
    [
        "_Evidence_Order",
        "_Conv_Risk_Order",
        "_Vol_Risk_Order",
        "Expected_RPU_Center"
    ],
    ascending=[
        True,
        True,
        True,
        False
    ]
)

pilot = pilot.drop(
    columns=[
        "_Evidence_Order",
        "_Conv_Risk_Order",
        "_Vol_Risk_Order"
    ]
)

pilot.to_csv(
    OUTPUT_DIR
    / "05_V13_PILOT_REVIEW_SHORTLIST.csv",
    index=False
)

display(
    pilot.head(30)
)


,Fee_Increase,Treated_Location,Treated_Lane,Treated_Price_Tier,Events,Change_Dates,Expected_RPU_Center,Expected_RPU_Low,Expected_RPU_High,RPU_Range_Interpretation,...,P_Volume_Down_GT_10pct,Segment_Confidence,Segment_Evidence_Strength,Portfolio_Evidence_Strength,Evidence_Strength,Median_R2,Median_MAE,Median_Direction_Accuracy,Prediction_Quality_Flag,Decision_Status


## 13. Final executive summary

In [14]:

summary_rows = []

for fee in SCENARIOS:

    d = final[
        final[
            "Fee_Increase"
        ].eq(fee)
    ]

    summary_rows.append({
        "Fee_Increase":
            fee,

        "Segments":
            len(d),

        "Locations":
            d[
                "Treated_Location"
            ].nunique(),

        "Median_Expected_RPU":
            d[
                "Expected_RPU_Center"
            ].median(),

        "Median_RPU_Low":
            d[
                "Expected_RPU_Low"
            ].median(),

        "Median_RPU_High":
            d[
                "Expected_RPU_High"
            ].median(),

        "Pct_Low_Conversion_Risk":
            (
                d[
                    "Conversion_Risk"
                ] == "Low"
            ).mean(),

        "Pct_Low_Volume_Risk":
            (
                d[
                    "Volume_Risk"
                ] == "Low"
            ).mean(),

        "Pct_High_or_Medium_Evidence":
            (
                d[
                    "Evidence_Strength"
                ].isin(
                    [
                        "High",
                        "Medium"
                    ]
                )
            ).mean(),

        "Pilot_Review_Segments":
            d[
                "Decision_Status"
            ].isin(
                [
                    "SUPPORTED FOR PILOT REVIEW",
                    "REVIEW WITH GUARDRAILS"
                ]
            ).sum(),
    })

executive = pd.DataFrame(
    summary_rows
)

executive.to_csv(
    OUTPUT_DIR
    / "06_V13_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("=" * 80)
print("V13 COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(executive)


V13 COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v13_outputs


,Fee_Increase,Segments,Locations,Median_Expected_RPU,Median_RPU_Low,Median_RPU_High,Pct_Low_Conversion_Risk,Pct_Low_Volume_Risk,Pct_High_or_Medium_Evidence,Pilot_Review_Segments
0,10,390,16,8.789637,-11.678715,32.370039,0.994872,1.000000,0.0,0
1,15,84,14,5.900824,-18.265413,23.252145,1.000000,1.000000,0.0,0
2,20,129,12,30.761922,7.503832,60.866812,1.000000,0.976744,0.0,0



## Files to share back

Please send:

1. `06_V13_EXECUTIVE_SUMMARY.csv`
2. `03_V13_PORTFOLIO_SCENARIO_SUMMARY.csv`
3. `04_V13_EVIDENCE_STRENGTH_SUMMARY.csv`
4. `05_V13_PILOT_REVIEW_SHORTLIST.csv`
5. `02_V13_SEGMENT_DECISION_TABLE.csv`

## How to interpret V13

V13 should be used as **decision support**, not an automated price setter.

The strongest use case is:

- identify segments with positive historical RPU evidence,
- screen out high Conversion / Volume risk,
- prioritize segments with enough independent historical support,
- use the shortlist for controlled pricing pilots,
- then feed actual pilot results back into the next model iteration.
